# Browsing Automation

## Resources:

- https://playwright.dev/python/docs/api/class-page 

## Test URLs
### Works on PlayWright: 
- https://jobs.workable.com/view/5Q8tVt6rfpWg4A8H8FkFiB/remote-ai-solutions-engineer-in-buenos-aires-at-hire-overseas

- https://webscraper.io/test-sites/e-commerce/allinone

- https://jobs.generalcatalyst.com/companies/sonarsource/jobs/74080968-ai-engineer 


### Not Work on PlayWright: 
- https://wellfound.com/jobs/4050171-senior-ai-engineer-agentic-devsecops-automation-j318 

In [2]:
from typing import Optional, List
from dataclasses import dataclass

@dataclass
class Job:
    title: str
    url: str
    text: Optional[str]
    searched_via: str
    low_quality: bool = False

In [3]:
from playwright.sync_api import sync_playwright

## Figure out a way to minimalizing noise of HTML tags the best 
Before passing to the LLM

In [32]:
from playwright.async_api import async_playwright
import asyncio
import re

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://jobs.generalcatalyst.com/companies/sonarsource/jobs/74080968-ai-engineer")
        
        desc_patterns = ["Description", "About the Role", "Job Description", "What You’ll Be Doing", "Why should I Apply"]

        for desc_pattern in desc_patterns:
            locator = page.get_by_role("heading", name=re.compile(desc_pattern, re.I))


            if await locator.count() > 0: 
                section = locator.first.locator("xpath=..")
                text = await section.inner_text()

                print(f"FOUND SECTION: {desc_pattern.upper()}")
                print(text)
                break
        
        await browser.close()

await run()

## Trials and Errors: 


### page.get_by_text
Inconsistent because it’s random of the location where the first selected text matches. It can’t pull out all the text stably all the time. 

In [27]:
from playwright.async_api import async_playwright
import asyncio
import re

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://www.builtinnyc.com/job/ai-engineer/8349500")
        
        desc_patterns = ["Description", "About the Role", "Job Description", "What You’ll Be Doing", "skills"]

        for desc_pattern in desc_patterns:
            locator = page.get_by_text(desc_pattern.lower(), exact=False)


            if await locator.count() > 0: 
                # go one level up in the DOM (parent node) with xpath
                section = locator.first.locator("xpath=..")
                text = await section.inner_text()

                print(f"FOUND SECTION: {desc_pattern.upper()}")
                print(text)
                break
        
        await browser.close()

await run()

FOUND SECTION: DESCRIPTION
The requirements listed in the job descriptions are guidelines. You don’t have to satisfy every requirement or meet every qualification listed. If your skills are transferable we would still love to hear from you.


In [5]:
low_quality_urls = ['https://jobs.lever.co/binance/?location=Australia%2C%20Brisbane', 'https://corporate.deeplearning.ai/courses/practical-multi-ai-agents-and-advanced-use-cases-with-crewai/lesson/aix1c/content-creation-at-scale', 'https://jobs.lever.co/atmosera/c5e21ae4-fcf3-4fe0-83df-a018a4f9f1b1', 'https://apply.workable.com/netguru/j/60648FBA5C/', 'https://github.com/TEN-framework/TEN-Agent', 'https://fastgpt.io/en', 'https://redirect.github.com/speedyapply/2026-AI-College-Jobs', 'https://github.com/ChatFAQ/ChatFAQ', 'https://www.qcloud.com/developer/article/2628717', 'https://jobs.lever.co/binance?department=Engineering', 'https://apply.workable.com/mlabs/j/CF4E6EC215/', 'https://dify.ai/', 'https://jobs.sequoiacap.com/jobs/hugging-face', 'https://jobs.ashbyhq.com/Pyyne/11176bae-e8bf-4163-9f07-949941f80c84', 'https://dailyaimail.news/news/career-ops-claude-code-job-search-open-source', 'https://apply.workable.com/tiger-analytics/j/39C8151486/', 'https://unsloth.ai/docs/models/qwen3.5/fine-tune', 'https://apply.workable.com/huggingface/', 'https://www.useparallel.com/app/candidate/job/674e4ffe79c5f73418602259', 'https://www.digit.in/features/general/he-used-claude-code-to-apply-to-over-700-jobs-and-got-hired-heres-how.html', 'https://www.wantedly.com/projects/2275794', 'https://jobs.lever.co/binance/fe2b0bf9-95bb-40c5-959b-f991278a1cbe', 'https://github.com/primeIntellect-ai/prime-rl', 'https://www.facebook.com/groups/573518645176483/posts/809888104872868/', 'https://jobs.lever.co/binance?%20Design=&department=Engineering', 'https://apply.workable.com/volga-partners/j/35FCA61135', 'https://jobs.ashbyhq.com/cohere/1fa01a03-9253-4f62-8f10-0fe368b38cb9']

### page.content() + BS4

In [40]:
from playwright.async_api import async_playwright
import asyncio
from bs4 import BeautifulSoup
from bs4.element import Tag
import time
import random

async def run():
    async with async_playwright() as p:
        await asyncio.sleep(random.randint(1, 3))
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://jobs.ashbyhq.com/cohere/1fa01a03-9253-4f62-8f10-0fe368b38cb9")
        # Take some time to wait for the dom to fully load
        await page.wait_for_timeout(3000)

        html = await page.content()

        if html:
            soup = BeautifulSoup(html, "lxml")
            footer = soup.find("footer")

            # Remove footer
            if footer:
                footer.decompose()

            # Remove junk tags
            for tag in soup.find_all(["nav", "aside"]):
                tag.decompose()

            # Remove the other messy HTML elements by classes and IDs
            # Commented this out because this filters out all content sometimes
            
            # keywords = ["footer", "cookie", "privacy", "nav", "menu", "header"]
            # for tag in list(soup.find_all(True)):

            #     if not isinstance(tag, Tag) or tag.attrs is None:
            #         continue
                    
            #     classes = " ".join(tag.get("class", []))
            #     id_ = tag.get("id", "")

            #     combined = f"{classes} {id_}".lower()

            #     if any(keyword in combined for keyword in keywords):
            #         tag.decompose()
            
                
            text = soup.get_text(separator="\n\n", strip=True)

        
        print(text)
        await asyncio.sleep(random.randint(3, 8))
        await browser.close()

await run()

Applied AI Engineer – Agentic Workflows @ Cohere

You need to enable JavaScript to run this app.

Applied AI Engineer – Agentic Workflows

Location

San Francisco; London; Montreal; New York; Toronto

Employment Type

Full time

Location Type

Remote

Department

Modeling

Applied-ML

Who are we?

Our mission is to scale intelligence to serve humanity. We’re training and deploying frontier models for developers and enterprises who are building AI systems to power magical experiences like content generation, semantic search, RAG, and agents. We believe that our work is instrumental to the widespread adoption of AI.

We obsess over what we build. Each one of us is responsible for contributing to increasing the capabilities of our models and the value they drive for our customers. We like to work hard and move fast to do what’s best for our customers.

Cohere is a team of researchers, engineers, designers, and more, who are passionate about their craft. Each person is one of the best in t

## Test aria_snapshot()

Result:
aria_snapshot requires PlayWright version above 1.59 which isn't competible with my current
Python version (3.13). It needs to downgrade my Python to 3.11 and ultimately, aria_snapshot isn't as good or production-ready as page.content() as well.

In [5]:
from playwright.async_api import async_playwright
import asyncio

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        await page.goto("https://webscraper.io/test-sites/e-commerce/allinone")
        
        snapshot = await page.aria_snapshot(mode="ai")

        print(snapshot)

        await browser.close()
        

await run()

AttributeError: 'Page' object has no attribute 'aria_snapshot'